In [1]:
import pandas as pd
import numpy as np
import os
import json
import ipywidgets as widgets
from IPython.display import display

In [2]:
player_input = widgets.Text(
    placeholder='Enter player name',
    description='Player:'
)

display(player_input)

Text(value='', description='Player:', placeholder='Enter player name')

In [3]:
if not player_input.value.strip():
    raise ValueError("Please enter a player name in the previous cell before proceeding.")

player_name = player_input.value.strip()

In [4]:
combined_data_shots = pd.read_excel(f'../../data/mens/{player_name}/combined.xlsx', sheet_name='Shots')
combined_data_points = pd.read_excel(f'../../data/mens/{player_name}/combined.xlsx', sheet_name='Points')
combined_data_games = pd.read_excel(f'../../data/mens/{player_name}/combined.xlsx', sheet_name='Games')
combined_data_sets = pd.read_excel(f'../../data/mens/{player_name}/combined.xlsx', sheet_name='Sets')
combined_data_stats = pd.read_excel(f'../../data/mens/{player_name}/combined.xlsx', sheet_name='Stats')

Helper Functions

In [5]:
DATA_X_MAX = 35.0
DATA_Y_MAX = 35.0
COURT_Y_OFFSET = 11.8872

FDATA_X_MAX = 29.0
FDATA_Y_MAX = 29.0
FCOURT_Y_OFFSET = 9

def normalize_orientation(hit_x, hit_y, bounce_x, bounce_y):
    if hit_y < 0:
        hit_x, hit_y, bounce_x, bounce_y = -hit_x, -hit_y, -bounce_x, -bounce_y
    if bounce_y > 0:
        bounce_y = -bounce_y
    return hit_x, hit_y, bounce_x, bounce_y

def scale_coords(df):
    df = df.copy()
    coords = df.apply(
        lambda r: pd.Series(
            normalize_orientation(r["Hit (x)"], r["Hit (y)"], r["Bounce (x)"], r["Bounce (y)"]),
            index=["hit_x", "hit_y", "bounce_x", "bounce_y"]
        ),
        axis=1
    )
    df[["hit_x", "hit_y", "bounce_x", "bounce_y"]] = coords

    near = df["Hit Side"] == "near"
    df.loc[near, "hit_x"]    = df.loc[near, "hit_x"]    * DATA_X_MAX
    df.loc[near, "hit_y"]    = (df.loc[near, "hit_y"]    + COURT_Y_OFFSET) * DATA_Y_MAX
    df.loc[near, "bounce_x"] = df.loc[near, "bounce_x"] * DATA_X_MAX
    df.loc[near, "bounce_y"] = (df.loc[near, "bounce_y"] + COURT_Y_OFFSET) * DATA_Y_MAX

    far = df["Hit Side"] == "far"
    df.loc[far, "hit_x"]    = df.loc[far, "hit_x"]    * FDATA_X_MAX
    df.loc[far, "hit_y"]    = (df.loc[far, "hit_y"]  - FCOURT_Y_OFFSET) * FDATA_Y_MAX
    df.loc[far, "bounce_x"] = df.loc[far, "bounce_x"] * FDATA_X_MAX
    df.loc[far, "bounce_y"] = (df.loc[far, "bounce_y"] - FCOURT_Y_OFFSET) * FDATA_Y_MAX

    # needs_flip = df["Hit (y)"] < 0
    # df.loc[needs_flip, "Hit Zone"]    = df.loc[needs_flip, "Hit Zone"].str.lower().map(FLIP_ZONE)
    # df.loc[needs_flip, "Bounce Zone"] = df.loc[needs_flip, "Bounce Zone"].str.lower().map(FLIP_ZONE)

    return df
 
 
def classify_bounce_side(bounce_x):
    """Classify where the ball is going: backhand side = bounce_x < 0."""
    if pd.isna(bounce_x):
        return np.nan
    return "Backhand Side" if bounce_x < 0 else "Forehand Side"
 
 
def get_previous_direction(df_all_shots, io_fh_df):
    rows = []
    for _, row in io_fh_df.iterrows():
        mask = (
            (df_all_shots["Point"]           == row["Point"]) &
            (df_all_shots["Game"]            == row["Game"]) &
            (df_all_shots["Set"]             == row["Set"]) &
            (df_all_shots["__source_file__"] == row["__source_file__"]) &
            (df_all_shots["Shot"]            < row["Shot"])
        )
        candidates = df_all_shots[mask]
        if not candidates.empty:
            prev = candidates.sort_values("Shot").iloc[-1]
            rows.append(prev.get("Direction", np.nan))
        else:
            rows.append("---")

    io_fh_df["prev_direction"] = rows
    return io_fh_df
 
def get_opponent_positions(df_all_shots, io_fh_df, opp_name):
    opp = df_all_shots[df_all_shots["Player"] == opp_name][
        ["Point", "Game", "Set", "__source_file__", "Shot", "Hit (x)", "Hit (y)"]
    ].copy()
    opp = opp.rename(columns={"Shot": "opp_shot", "Hit (x)": "opp_hit_x_raw", "Hit (y)": "opp_hit_y_raw"})
    # Opponent is on the opposite end from the player, so flip their coordinates
    # to match the normalized player orientation (player always hits from positive-y side,
    # so opponent is always on the negative-y side — negate both axes to align).
    opp["opp_hit_x"] = -opp["opp_hit_x_raw"]
    opp["opp_hit_y"] = -opp["opp_hit_y_raw"]
 
    rows = []
    for _, row in io_fh_df.iterrows():
        mask = (
            (opp["Point"] == row["Point"]) &
            (opp["Game"]  == row["Game"])  &
            (opp["Set"]   == row["Set"])   &
            (opp["__source_file__"] == row["__source_file__"]) &
            (opp["opp_shot"] < row["Shot"])
        )
        candidates = opp[mask]
        if not candidates.empty:
            last = candidates.sort_values("opp_shot").iloc[-1]
            rows.append({"Point": row["Point"], "Game": row["Game"], "Set": row["Set"],
                         "__source_file__": row["__source_file__"], "Shot": row["Shot"],
                         "opp_hit_x": last["opp_hit_x"], "opp_hit_y": last["opp_hit_y"]})
        else:
            rows.append({"Point": row["Point"], "Game": row["Game"], "Set": row["Set"],
                         "__source_file__": row["__source_file__"], "Shot": row["Shot"],
                         "opp_hit_x": np.nan, "opp_hit_y": np.nan})
 
    opp_pos = pd.DataFrame(rows)
    return pd.merge(io_fh_df, opp_pos, on=["Point", "Game", "Set", "__source_file__", "Shot"], how="left")

In [6]:
combined = pd.merge(
    combined_data_shots,
    combined_data_points[["Point", "Game", "Set", "Point Winner", "Match Server", "__source_file__"]],
    on=["Point", "Game", "Set", "__source_file__"],
    how="left"
)
 
io_fh = combined[
    (combined["Player"]    == player_name) &
    (combined["Stroke"]    == "Forehand") &
    (combined["Direction"].str.lower().str.contains("inside"))
].copy()
 
print(f"Total inside-out forehand shots: {len(io_fh)}")

Total inside-out forehand shots: 106


In [7]:
io_fh = scale_coords(io_fh)

io_fh["bounce_side"]    = io_fh["bounce_x"].apply(classify_bounce_side)
io_fh = get_previous_direction(combined_data_shots, io_fh)

In [8]:
io_fh_bh = io_fh[io_fh["bounce_side"] == "Backhand Side"].copy()

print(f"  Forehand shots going to Backhand Side: {len(io_fh_bh)}")
print(f"  (Excluded: shots going to Forehand Side: {len(io_fh[io_fh['bounce_side'] == 'Forehand Side'])})")


  Forehand shots going to Backhand Side: 53
  (Excluded: shots going to Forehand Side: 53)


In [9]:
all_players = combined_data_shots["Player"].dropna().unique()
opponents   = [p for p in all_players if p != player_name]
print(f"Opponents found: {opponents}")
 
if opponents:
    opponent_name = opponents[0]
    io_fh    = get_opponent_positions(combined_data_shots, io_fh, opponent_name)
    io_fh_bh = io_fh[io_fh["bounce_side"] == "Backhand Side"].copy()
    print(f"Opponent positions added using: {opponent_name}")
else:
    print("Warning: No opponent found — skipping opponent position step.")

Opponents found: ['Alejandro Arcila', 'Stian Klaasen', 'Matias Ponce de Leon', 'Gustavo Ribeiero']
Opponent positions added using: Alejandro Arcila


In [10]:
io_fh_bh = io_fh[io_fh["bounce_side"] == "Backhand Side"].copy()


In [11]:
def print_summary(df, label):
    total = len(df)
    if total == 0:
        print(f"\n{label}: No shots found.")
        return
 
    print(f"\n{'='*52}")
    print(f"  {label}  (n={total})")
    print(f"{'='*52}")
 
print_summary(io_fh,    "All Inside-Out Forehands")
print_summary(io_fh_bh, "IO-FH Going to Backhand Side")


  All Inside-Out Forehands  (n=108)

  IO-FH Going to Backhand Side  (n=54)


In [12]:
def print_hit_location(df, label):
    if len(df) == 0:
        return
    total = len(df)
    behind = df[df["hit_y"] < 0]
    inside = df[df["hit_y"] >= 0]
    print(f"\n{label} — Hit Location:")
    print(f"  hit_x  mean={df['hit_x'].mean():.1f}  median={df['hit_x'].median():.1f}  "
          f"std={df['hit_x'].std():.1f}  range=[{df['hit_x'].min():.1f}, {df['hit_x'].max():.1f}]")
    print(f"  hit_y  mean={df['hit_y'].mean():.1f}  median={df['hit_y'].median():.1f}  "
          f"std={df['hit_y'].std():.1f}  range=[{df['hit_y'].min():.1f}, {df['hit_y'].max():.1f}]")
    print(f"  Behind baseline (defensive): {len(behind)} ({round(len(behind)/total*100,1)}%)")
    print(f"  Inside/on baseline:          {len(inside)} ({round(len(inside)/total*100,1)}%)")
 
print_hit_location(io_fh_bh, "IO-FH Going to Backhand Side")

threshold_x = io_fh_bh["hit_x"].median() if len(io_fh_bh) > 0 else 0.0
print(f"\nInside-Out Forehand Zone Threshold (median hit_x): {threshold_x:.1f}")


IO-FH Going to Backhand Side — Hit Location:
  hit_x  mean=40.4  median=43.8  std=46.7  range=[-104.9, 118.6]
  hit_y  mean=463.5  median=462.8  std=29.7  range=[405.7, 563.1]
  Behind baseline (defensive): 0 (0.0%)
  Inside/on baseline:          54 (100.0%)

Inside-Out Forehand Zone Threshold (median hit_x): 43.8


In [13]:
def build_records(df):
    records = []
    for _, r in df.iterrows():
        rec = {
            "pointNumber":   int(r["Point"])  if not pd.isna(r["Point"])  else None,
            "game":          int(r["Game"])   if not pd.isna(r["Game"])   else None,
            "set":           int(r["Set"])    if not pd.isna(r["Set"])    else None,
            "shotNumber":    int(r["Shot"])   if not pd.isna(r["Shot"])   else None,
            "hit_x":         round(float(r["hit_x"]),    3) if not pd.isna(r["hit_x"])    else None,
            "hit_y":         round(float(r["hit_y"]),    3) if not pd.isna(r["hit_y"])    else None,
            "bounce_x":      round(float(r["bounce_x"]), 3) if not pd.isna(r["bounce_x"]) else None,
            "bounce_y":      round(float(r["bounce_y"]), 3) if not pd.isna(r["bounce_y"]) else None,
            "opp_hit_x":     round(float(r["opp_hit_x"]), 3) if "opp_hit_x" in r and not pd.isna(r.get("opp_hit_x", np.nan)) else None,
            "opp_hit_y":     round(float(r["opp_hit_y"]), 3) if "opp_hit_y" in r and not pd.isna(r.get("opp_hit_y", np.nan)) else None,
            "bounceSide":    r["bounce_side"],
            "ballDirection": r["prev_direction"],
            "sourceFile":    str(r["__source_file__"])
        }
        records.append(rec)
    return records
 
 
def build_summary_json(df, records, threshold):
    total = len(df)
    if total == 0:
        return {}
 
    return {
        "playerName":     player_name,
        "totalShots":     total,
        "directionBreakdown": {k: {"count": int(v), "pct": round(v/total*100,1)}
                                for k, v in df["prev_direction"].value_counts().items()},
        "hitLocation": {
            "mean_x":    round(float(df["hit_x"].mean()), 2),
            "mean_y":    round(float(df["hit_y"].mean()), 2),
            "median_x":  round(float(df["hit_x"].median()), 2),
            "threshold_x": round(float(threshold), 2)
        },
        "shots": records
    }
 
 
all_records = build_records(io_fh)
bh_records  = build_records(io_fh_bh)
 
with open(f"data/insideout_fh_all.json", "w") as f:
    json.dump(build_summary_json(io_fh, all_records, threshold_x), f, indent=4)
 
with open(f"data/insideout_fh_bh_side.json", "w") as f:
    json.dump(build_summary_json(io_fh_bh, bh_records, threshold_x), f, indent=4)
 
print(f"\nSaved: data/insideout_fh_all.json")
print(f"Saved: data/insideout_fh_bh_side.json")


Saved: data/insideout_fh_all.json
Saved: data/insideout_fh_bh_side.json


In [14]:
export_cols = [
    "Point", "Game", "Set", "Shot", "__source_file__",
    "hit_x", "hit_y", "bounce_x", "bounce_y",
    "bounce_side", "ball_direction"
]
if "opp_hit_x" in io_fh.columns:
    export_cols += ["opp_hit_x", "opp_hit_y"]
export_cols = [c for c in export_cols if c in io_fh.columns]
 
io_fh[export_cols].to_csv(f"data/{player_name}_insideout_fh.csv", index=False)
print(f"Saved: data/{player_name}_insideout_fh.csv")
 
# Summary CSV
summary_rows = []
for label, df_s in [("All IO-FH", io_fh), ("Backhand Side", io_fh_bh)]:
    if len(df_s) == 0:
        continue
    total = len(df_s)
    summary_rows.append({
        "section":             label,
        "total_shots":         total,
        "cross_court_pct":     f"{round(len(df_s[df_s['prev_direction']=='Cross Court'])/total*100,1)}%",
        "down_the_line_pct":   f"{round(len(df_s[df_s['prev_direction']=='Down the Line'])/total*100,1)}%",
        "mean_hit_x":          round(float(df_s["hit_x"].mean()), 2),
        "mean_hit_y":          round(float(df_s["hit_y"].mean()), 2),
        "threshold_x":         round(float(threshold_x), 2),
    })
 
pd.DataFrame(summary_rows).to_csv(f"data/{player_name}_insideout_fh_summary.csv", index=False)
print(f"Saved: data/{player_name}_insideout_fh_summary.csv")

Saved: data/Stian Klaassen_insideout_fh.csv
Saved: data/Stian Klaassen_insideout_fh_summary.csv


In [15]:
def classify_zone(df):
    x = df['x_coord']
    
    if x >= 0:
        side = 'Forehand'
    elif x < 0:
        side = 'Backhand'
    else:
        side = np.nan
    
    return pd.Series({'side': side})

In [16]:
combined_data_shots

player_shots = combined_data_shots[
    (combined_data_shots['Player'] == player_name) &
    (combined_data_shots['Stroke'] == 'Forehand') &
    (combined_data_shots['Direction'] == 'inside out')
].copy()

player_shots


,Player,Shot,Type,Stroke,Spin,Speed (MPH),Point,Game,Set,Bounce Depth,...,Hit Side,Hit (x),Hit (y),Hit (z),Direction,Result,Favorited,Start Time,Video Time,__source_file__
941,Stian Klaassen,2,first_return,Forehand,Flat,51.037544,13,3,1,deep,...,near,-2.171875,-1.503274,1.072266,inside out,In,False,23:01:18,462.220001,StianKlaasen_LSU_10_31_25.xlsx
979,Stian Klaassen,5,in_play,Forehand,Topspin,62.294216,20,4,1,deep,...,far,1.666992,25.880663,0.700684,inside out,In,False,23:06:30,774.849976,StianKlaasen_LSU_10_31_25.xlsx
1045,Stian Klaassen,2,first_return,Forehand,Topspin,41.318577,34,7,1,short,...,near,-2.068359,-0.498392,1.571289,inside out,In,False,23:16:08,1352.089966,StianKlaasen_LSU_10_31_25.xlsx
1071,Stian Klaassen,5,in_play,Forehand,Topspin,50.936211,40,8,1,short,...,far,3.113281,24.661913,1.626953,inside out,In,False,23:20:46,1630.510010,StianKlaasen_LSU_10_31_25.xlsx
1189,Stian Klaassen,3,serve_plus_one,Forehand,Topspin,63.640686,55,10,2,deep,...,near,-0.621094,-1.757669,0.953613,inside out,In,False,23:36:17,2561.909912,StianKlaasen_LSU_10_31_25.xlsx
1191,Stian Klaassen,5,in_play,Forehand,Topspin,55.434486,55,10,2,short,...,near,-2.339844,-1.572122,0.560059,inside out,In,False,23:36:20,2564.909912,StianKlaasen_LSU_10_31_25.xlsx
1217,Stian Klaassen,2,first_return,Forehand,Flat,37.268608,59,11,2,short,...,far,1.768555,25.599413,1.200195,inside out,Net,False,23:40:08,2792.399902,StianKlaasen_LSU_10_31_25.xlsx
1236,Stian Klaassen,4,return_plus_one,Forehand,Topspin,50.975506,61,11,2,deep,...,far,1.432617,25.380663,1.848633,inside out,In,False,23:41:35,2879.229980,StianKlaasen_LSU_10_31_25.xlsx
1238,Stian Klaassen,6,in_play,Forehand,Slice,37.535770,61,11,2,short,...,far,2.373047,26.052538,0.648926,inside out,Net,False,23:41:38,2882.030029,StianKlaasen_LSU_10_31_25.xlsx
1268,Stian Klaassen,20,in_play,Forehand,Topspin,51.237560,64,11,2,deep,...,far,2.566406,26.083788,0.753418,inside out,In,False,23:43:59,3023.300049,StianKlaasen_LSU_10_31_25.xlsx


In [17]:
nan_rows = io_fh[io_fh['prev_direction'].isna()]
print(nan_rows[['Point', 'Shot', 'Hit Zone', 'Bounce Zone', '__source_file__']].head(30))

Empty DataFrame
Columns: [Point, Shot, Hit Zone, Bounce Zone, __source_file__]
Index: []


In [18]:
print(io_fh["prev_direction"].unique())

['down the T' 'down the line' 'inside out' 'cross court' 'out wide'
 'inside in' '---']
